<a href="https://colab.research.google.com/github/HABalyze/agente_auditor/blob/main/optimizar_reglas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Optimizador de Reglas.json
### Generación y validación de keywords

Con este notebook se busca analizar los casos y sugerir keywords de activación para los diferentes controles
en base a las `reglas.json`.
El usuario podra aceptar, editar o rechazar cada sugerencia antes de guardar.

### 0. Instalación de dependencias

In [ ]:
# Descargar el repositorio directamente desde GitHub
!git clone https://github.com/HABalyze/agente_auditor.git

# Cambiar el directorio de trabajo a la carpeta del proyecto
%cd agente_auditor

In [ ]:
pip install ipywidgets nltk

### 1. Importaciones y designación de las rutas

In [ ]:
import json
import re
from pathlib import Path
from collections import Counter
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import nltk
import copy
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize
nltk.download('stopwords')
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)


BASE_DIR = Path(".")
CASOS_PATH = BASE_DIR / "data" / "casos.json"
REGLAS_PATH = BASE_DIR / "config" / "reglas.json"

print("Cargas Correctas")

### 2. Carga de Archivos

In [ ]:
def cargar_json(ruta: Path, nombre: str):
    with open(ruta, "r", encoding="utf-8") as f:
        return json.load(f)

casos  = cargar_json(CASOS_PATH,  "casos.json")
reglas = cargar_json(REGLAS_PATH, "reglas.json")

print(f"Casos cargados    : {len(casos)}")
print(f"Controles activos : {len(reglas['controles'])}")
print(f"Controles         : {list(reglas['controles'].keys())}")

### 3. Validación de los Casos

Se revisan los textos reales para entender que palabras clave pueden existir.

In [ ]:
for caso in casos:
    print(f"\n{'='*60}")
    print(f"Caso {caso['id_caso']}")
    print(f"Contexto RAG : {caso['contexto_rag']}")
    print(f"Respuesta B  : {caso['respuesta_agent_b']}")

### 4. Motor de Validación y Extracción de KeyWords

Extraer terminos relevantes de los contextos RAG dentro de los casos, filtrando posbiles `stopwords`

In [ ]:
STOPWORDS = set(stopwords.words('spanish'))

def extraer_keywords(texto: str, top_n: int = 8) -> list:
    # Solo bigramas con contenido semántico real
    bigramas = re.findall(r'[a-záéíóúüñ]+\s+[a-záéíóúüñ]+', texto.lower())
    bigramas = [
        b for b in bigramas
        if not all(w in STOPWORDS for w in b.split())
        and not any(w in STOPWORDS for w in b.split())  # ninguna palabra del bigrama es stopword
        and len(b) > 6
    ]
    palabras = re.findall(r'[a-záéíóúüñA-ZÁÉÍÓÚÜÑ]{5,}', texto)
    palabras = [p.lower() for p in palabras if p.lower() not in STOPWORDS and len(p) > 4]
    mayus    = re.findall(r'[A-Z_]{4,}', texto)
    todos    = bigramas + palabras + mayus
    return [term for term, _ in Counter(todos).most_common(top_n)]


def separar_contextos_por_tipo(casos: list, reglas: dict) -> dict:
    """
    Separa los contextos por tipo de control usando los regex
    que ya existen en reglas.json — sin hardcode nuevo.
    """
    patron_monto = reglas["extraccion"]["montos_usd"]["regex"]
    patron_edad  = reglas["extraccion"]["edad"]["regex"]

    contextos_por_tipo = {
        "deteccion_urgencia":   [],
        "comparacion_numerica": [],
        "deteccion_derivacion": [],
    }

    for caso in casos:
        ctx = caso["contexto_rag"]
        ctx_lower = ctx.lower()

        # Tiene monto o edad → limite numerico
        tiene_monto = bool(re.search(patron_monto, ctx))
        tiene_edad  = bool(re.search(patron_edad, ctx, re.IGNORECASE))
        if tiene_monto or tiene_edad:
            contextos_por_tipo["comparacion_numerica"].append(ctx)

        # Tiene palabras de alerta en mayúsculas o bloqueo → urgencia
        tiene_alerta = bool(re.search(r'[A-Z_]{4,}|bloquear|restrictiva|lavado', ctx))
        if tiene_alerta:
            contextos_por_tipo["deteccion_urgencia"].append(ctx)

        # Tiene historial o sospecha → derivacion
        tiene_sospecha = bool(re.search(r'sospecha|historial|siniestros', ctx_lower))
        if tiene_sospecha:
            contextos_por_tipo["deteccion_derivacion"].append(ctx)

    return contextos_por_tipo


def sugerir_todo(casos: list, reglas: dict) -> dict:
    contextos_por_tipo = separar_contextos_por_tipo(casos, reglas)

    sugerencias = {}
    for nombre, ctrl in reglas["controles"].items():
        tipo = ctrl.get("tipo", "")
        textos_ctx = contextos_por_tipo.get(tipo, [])

        # Casos donde este control aplica
        casos_relevantes = [
            c for c in casos
            if any(ctx == c["contexto_rag"] for ctx in textos_ctx)
        ]

        señales_correctas = ["analista", "no puedo procesar", "no es posible", "especialista", "contactará", "derivado"]
        señales_indebidas = ["aprobado", "exitoso", "emitida", "procesado", "autorizado", "correctamente"]

        # Respuestas correctas — solo de casos relevantes para este control
        resp_correctas = " ".join([
            c["respuesta_agent_b"] for c in casos_relevantes
            if any(s in c["respuesta_agent_b"].lower() for s in señales_correctas)
            and not any(s in c["respuesta_agent_b"].lower() for s in señales_indebidas)
        ])

        # Respuestas indebidas — solo de casos relevantes para este control
        resp_indebidas = " ".join([
            c["respuesta_agent_b"] for c in casos_relevantes
            if any(s in c["respuesta_agent_b"].lower() for s in señales_indebidas)
            and not any(s in c["respuesta_agent_b"].lower() for s in señales_correctas)
        ])

        sugerencias[nombre] = {
            "activacion": extraer_keywords(" ".join(textos_ctx)) if textos_ctx else [],
            "correctas":  extraer_keywords(resp_correctas) if "keywords_accion_correcta" in ctrl and resp_correctas else [],
            "indebidas":  extraer_keywords(resp_indebidas) if "keywords_aprobacion_indebida" in ctrl and resp_indebidas else [],
        }

    # Compartir keywords_accion_correcta entre controles que detectan derivacion
    correctas_compartidas = set()
    for nombre, s in sugerencias.items():
        if s.get("correctas"):
            correctas_compartidas.update(s["correctas"])

    for nombre, ctrl in reglas["controles"].items():
        if "keywords_accion_correcta" in ctrl and not sugerencias[nombre]["correctas"]:
            sugerencias[nombre]["correctas"] = list(correctas_compartidas)

    return sugerencias


sugerencias = sugerir_todo(casos, reglas)
print("Extracción completada ✅")
for ctrl, s in sugerencias.items():
    print(f"\n{ctrl}")
    print(f"  activacion : {s['activacion']}")
    print(f"  correctas  : {s['correctas']}")
    print(f"  indebidas  : {s['indebidas']}")

### 5. Tabla visual de Sugerencias

Se crea una vista comparativa de keywords actuales en `reglas.json` vs sugeridas por el control anterior.

Se itera para ir actualizando las reglas.

In [ ]:
filas_html = ""
for nombre, ctrl in reglas["controles"].items():
    s = sugerencias.get(nombre, {})

    act_kw   = ctrl.get("keywords_activacion", [])
    act_corr = ctrl.get("keywords_accion_correcta", [])
    act_ind  = ctrl.get("keywords_aprobacion_indebida", [])

    def render(actuales, sugeridas, color):
        nuevas = [k for k in sugeridas if k not in actuales]
        act_html = ", ".join(f'<code>{k}</code>' for k in actuales) if actuales else '<i>ninguna</i>'
        sug_html = ", ".join(f'<code style="color:{color}">{k}</code>' for k in nuevas) if nuevas else '<i>sin cambios</i>'
        return act_html, sug_html

    kw_act,   kw_sug   = render(act_kw,   s.get("activacion", []), "#3498db")
    corr_act, corr_sug = render(act_corr, s.get("correctas",  []), "#2ecc71")
    ind_act,  ind_sug  = render(act_ind,  s.get("indebidas",  []), "#e74c3c")

    filas_html += f"""
    <tr style="background:#0d0d1a">
        <td colspan="3" style="padding:8px 10px;font-weight:600;color:white">{nombre} <span style="color:#f39c12;font-size:11px">[{ctrl['tipo']}]</span></td>
    </tr>
    <tr>
        <td style="padding:6px 10px;color:#3498db;font-size:12px"> Activación</td>
        <td style="padding:6px 10px;font-size:12px">{kw_act}</td>
        <td style="padding:6px 10px;font-size:12px">{kw_sug}</td>
    </tr>
    <tr>
        <td style="padding:6px 10px;color:#2ecc71;font-size:12px"> Acción correcta</td>
        <td style="padding:6px 10px;font-size:12px">{corr_act}</td>
        <td style="padding:6px 10px;font-size:12px">{corr_sug}</td>
    </tr>
    <tr>
        <td style="padding:6px 10px;color:#e74c3c;font-size:12px;border-bottom:2px solid #333"> Aprobación indebida</td>
        <td style="padding:6px 10px;font-size:12px;border-bottom:2px solid #333">{ind_act}</td>
        <td style="padding:6px 10px;font-size:12px;border-bottom:2px solid #333">{ind_sug}</td>
    </tr>"""

tabla = f"""
<table style="width:100%;border-collapse:collapse;font-size:13px">
  <thead>
    <tr style="background:#1a1a2e;color:white">
      <th style="padding:10px;text-align:left">Campo</th>
      <th style="padding:10px;text-align:left">Actuales</th>
      <th style="padding:10px;text-align:left">Sugeridas</th>
    </tr>
  </thead>
  <tbody>{filas_html}</tbody>
</table>
"""
display(HTML(tabla))

### 6. Editor de Reglas

Seleecionar las keywords que se consideran aceptar por controy y edicar si es necesario.

In [ ]:
widgets_por_control = {}

for nombre, ctrl in reglas["controles"].items():
    s = sugerencias.get(nombre, {})

    titulo = widgets.HTML(f"""
    <div style="margin-top:20px;padding:10px 14px;background:#1a1a2e;border-radius:6px;border-left:4px solid #3498db">
        <b style="color:white;font-size:14px">{nombre}</b>
        <span style="color:#f39c12;font-size:12px"> [{ctrl['tipo']}]</span><br>
        <span style="color:#aaa;font-size:11px">{ctrl['descripcion']}</span>
    </div>""")
    display(titulo)

    def make_section(label, color, keywords_sugeridas, keywords_actuales):
        # Solo mostrar keywords que NO están ya en reglas.json
        nuevas = [kw for kw in keywords_sugeridas if kw not in keywords_actuales]

        if not nuevas:
            lbl = widgets.HTML(
                f"<span style='font-size:12px;color:{color};margin-top:8px;display:block'>"
                f"{label} — <i style='color:#aaa'>sin nuevas sugerencias</i></span>"
            )
            manual = widgets.Text(
                placeholder="Agregar manualmente (separar por coma)",
                layout=widgets.Layout(width="500px", margin="4px 0 12px 0")
            )
            display(lbl, manual)
            return {"checks": [], "manual": manual}

        lbl = widgets.HTML(
            f"<span style='font-size:12px;color:{color};margin-top:8px;display:block'>{label}</span>"
        )
        checks = [
            widgets.Checkbox(
                value=False,
                description=kw,
                style={"description_width": "initial"},
                layout=widgets.Layout(width="280px")
            )
            for kw in nuevas
        ]
        caja   = widgets.HBox(checks, layout=widgets.Layout(flex_flow="row wrap"))
        manual = widgets.Text(
            placeholder="Agregar manualmente (separar por coma)",
            layout=widgets.Layout(width="500px", margin="4px 0 12px 0")
        )
        display(lbl, caja, manual)
        return {"checks": checks, "manual": manual}

    # Siempre — keywords de activación
    w_kw = make_section(
        "🔵 Keywords de Activación", "#3498db",
        s.get("activacion", []),
        ctrl.get("keywords_activacion", [])
    )

    # Solo si el control tiene accion correcta
    w_corr = {}
    if "keywords_accion_correcta" in ctrl:
        w_corr = make_section(
            "🟢 Keywords Acción Correcta", "#2ecc71",
            s.get("correctas", []),
            ctrl.get("keywords_accion_correcta", [])
        )

    # Solo si el control tiene aprobacion indebida
    w_ind = {}
    if "keywords_aprobacion_indebida" in ctrl:
        w_ind = make_section(
            "🔴 Keywords Aprobación Indebida", "#e74c3c",
            s.get("indebidas", []),
            ctrl.get("keywords_aprobacion_indebida", [])
        )

    widgets_por_control[nombre] = {
        "kw":   w_kw,
        "corr": w_corr,
        "ind":  w_ind,
        "ctrl": ctrl
    }

print("\nEditor cargado. Ajusta las selecciones y corre la celda 7 para guardar.")

### 7. Guardar en `reglas.json`

Se consolidas las selecciones del paso anterior y se actualiza archivo de configuración.

In [ ]:
reglas_actualizadas = copy.deepcopy(reglas)
resumen = []

for nombre, w in widgets_por_control.items():
    ctrl = reglas["controles"][nombre]

    # keywords_activacion
    seleccionadas = [cb.description for cb in w["kw"]["checks"] if cb.value]
    manuales      = [k.strip() for k in w["kw"]["manual"].value.split(",") if k.strip()]
    existentes    = set(ctrl.get("keywords_activacion", []))
    reglas_actualizadas["controles"][nombre]["keywords_activacion"] = list(existentes | set(seleccionadas) | set(manuales))

    # keywords_accion_correcta
    if "keywords_accion_correcta" in ctrl and w.get("corr"):
        selec_corr      = [cb.description for cb in w["corr"]["checks"] if cb.value]
        manuales_corr   = [k.strip() for k in w["corr"]["manual"].value.split(",") if k.strip()]
        existentes_corr = set(ctrl.get("keywords_accion_correcta", []))
        reglas_actualizadas["controles"][nombre]["keywords_accion_correcta"] = list(existentes_corr | set(selec_corr) | set(manuales_corr))

    # keywords_aprobacion_indebida
    if "keywords_aprobacion_indebida" in ctrl and w.get("ind"):
        selec_ind      = [cb.description for cb in w["ind"]["checks"] if cb.value]
        manuales_ind   = [k.strip() for k in w["ind"]["manual"].value.split(",") if k.strip()]
        existentes_ind = set(ctrl.get("keywords_aprobacion_indebida", []))
        reglas_actualizadas["controles"][nombre]["keywords_aprobacion_indebida"] = list(existentes_ind | set(selec_ind) | set(manuales_ind))

    resumen.append((nombre, reglas_actualizadas["controles"][nombre]))

# Guardar
with open(REGLAS_PATH, "w", encoding="utf-8") as f:
    json.dump(reglas_actualizadas, f, ensure_ascii=False, indent=2)

print("reglas.json actualizado ✅\n")
print(f"{'─'*60}")
for nombre, ctrl in resumen:
    print(f"{nombre}:")
    print(f"  keywords_activacion         : {ctrl.get('keywords_activacion', [])}")
    print(f"  keywords_accion_correcta    : {ctrl.get('keywords_accion_correcta', [])}")
    print(f"  keywords_aprobacion_indebida: {ctrl.get('keywords_aprobacion_indebida', [])}\n")
print(f"{'─'*60}")